In [ ]:
import os  
import re  
import glob  
import io  
import pandas as pd  
from dotenv import load_dotenv  
import msoffcrypto  
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl") 

In [5]:
# ---------------- function to read protected excel ----------------

def read_protected_excel(file_path, password, sheet_name, skiprows):  
    """Read a single sheet from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    workbook.seek(0)  
    df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
    return df

# ---------------- config ----------------

base_paths = [  
    'L:/2026 Pilar Plant Files/2026 Manpower Detail/',  
]

load_dotenv()  
password = os.getenv("SM_REPORTS_PW")

sheet_name = 'MTD LDR'  
skiprows = 5  
required_names = ['LENOX SAL', 'MEETH SAL', 'GVCCC SAL']  
start_date = pd.to_datetime('2023-01-01')

# ---------------- gather all .xlsm files ----------------

all_files = []  
for base_path in base_paths:  
    if not os.path.isdir(base_path):  
        print(f"WARNING: path not found, skipping -> {base_path}")  
        continue  
    found = glob.glob(os.path.join(base_path, '*.xlsm'))  
    print(f"{len(found):4d} .xlsm files in {base_path}")  
    all_files.extend(found)

# ---------------- filter + find latest file per month per site ----------------

candidates = []

for f in all_files:  
    filename = os.path.basename(f)

    if filename.startswith('~$'):  # Excel lock/temp file  
        continue

    upper_name = filename.upper()  
    matched_name = next((n for n in required_names if n.upper() in upper_name), None)  
    if matched_name is None:  
        continue

    site_code = matched_name.split(' ')[0]

    date_match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)  
    if not date_match:  
        print(f"  WARNING: Could not parse date from {filename}")  
        continue

    file_date = pd.to_datetime(date_match.group(1), errors='coerce')  
    if pd.isna(file_date):  
        print(f"  WARNING: Bad date '{date_match.group(1)}' in {filename}")  
        continue

    if file_date < start_date:  
        continue

    candidates.append((f, file_date, site_code))

# pick the latest file per (year, month, site)  
latest_files_map = {}

for f_path, f_date, site_code in candidates:  
    key = (f_date.year, f_date.month, site_code)  
    if key not in latest_files_map or f_date > latest_files_map[key][1]:  
        latest_files_map[key] = (f_path, f_date, site_code)

files_to_process = sorted(  
    latest_files_map.values(),  
    key=lambda x: (x[2], x[1])  # sort by site, then date  
)

print(f"\nFiles found (last of each month, per site, on/after {start_date.date()}): {len(files_to_process)}")  
for f, d, site in files_to_process:  
    print(f"  {os.path.basename(f)} (date: {d.date()} | site: {site})")

if not files_to_process:  
    raise SystemExit("No files matched. Check base_paths / required_names / start_date.")

# ---------------- read + stack MTD LDR ----------------

all_mtd_ldr = []

for file_path, file_date, site_code in files_to_process:  
    print(f"\nReading MTD LDR | Site: {site_code} | Date: {file_date.date()}")

    try:  
        df = read_protected_excel(file_path, password, sheet_name, skiprows)  
        df['Report_Date'] = file_date  
        df['BU'] = site_code  
        all_mtd_ldr.append(df)  
    except Exception as e:  
        print(f"  ERROR: {e}")

# ---------------- combine + clean ----------------

mtd_ldr_combined = pd.concat(all_mtd_ldr, ignore_index=True)

mtd_ldr_combined.columns = (  
    mtd_ldr_combined.columns.str.strip()  
    .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
    .str.replace(r'_+', '_', regex=True)  
    .str.strip('_')  
    .str.lower()  
)

print(f"\n--- Summary ---")  
print(f"MTD LDR: {mtd_ldr_combined.shape}")  
print(f"Sites found: {mtd_ldr_combined['bu'].unique()}")  

 150 .xlsm files in L:/2026 Pilar Plant Files/2026 Manpower Detail/

Files found (last of each month, per site, on/after 2023-01-01): 30
  GVCCC SALAERN GVCCC_EXEC 2026-01-31.xlsm (date: 2026-01-31 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-02-28.xlsm (date: 2026-02-28 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-03-31.xlsm (date: 2026-03-31 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-04-30.xlsm (date: 2026-04-30 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-05-31.xlsm (date: 2026-05-31 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-06-30.xlsm (date: 2026-06-30 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-07-31.xlsm (date: 2026-07-31 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-08-31.xlsm (date: 2026-08-31 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-09-30.xlsm (date: 2026-09-30 | site: GVCCC)
  GVCCC SALAERN GVCCC_EXEC 2026-10-17.xlsm (date: 2026-10-17 | site: GVCCC)
  LENOX SALAERN LENOX_EXEC 2026-01-31.xlsm (date: 2026-01-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 

In [17]:
folder_path = 'L:/2026 Pilar Plant Files/Position Control/'

site_names = ['LENOX', 'MEETH', 'GVCCC']

# Get all excel files, excluding temp lock files (~$)  
all_excel_files = glob.glob(os.path.join(folder_path, '*.xlsx'))  
all_excel_files = [  
    f for f in all_excel_files  
    if not os.path.basename(f).startswith('~$')  
]

all_data = []

for site in site_names:  
    # Filter files for this site (case-insensitive)  
    site_files = [  
        f for f in all_excel_files  
        if site in os.path.basename(f).upper()  
    ]

    # Extract date from each filename  
    dated_files = []  
    for file in site_files:  
        file_name = os.path.basename(file)  
        match = re.search(r'(\d{4}-\d{2}-\d{2})', file_name)  
        if match:  
            file_date = pd.to_datetime(match.group(1), errors='coerce')  
            if pd.notna(file_date):  
                dated_files.append((file, file_date))  
        else:  
            print(f"WARNING: Could not parse date from {file_name}")

    if not dated_files:  
        print(f"WARNING: No valid dated files found for {site}")  
        continue

    # Get the most recent file for this site  
    most_recent_file, most_recent_date = max(dated_files, key=lambda x: x[1])  
    file_name = os.path.basename(most_recent_file)

    print(f"\nMost recent file for {site}: {file_name} (date: {most_recent_date.date()})")

    try:  
        df = pd.read_excel(most_recent_file, sheet_name='PositionDetail', skiprows=12)  

        df['Date'] = most_recent_date  
        df['BU'] = site  
        df = df[['Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title', 'Job Function / Family', 'Posn Number',   
                                        'Posn Status', 'Filled/ Open', 'Incumbent Name', 'Emplid', 'Incumbent Status', 'STD Hrs', 
                                        'Reg/ Temp', 'Posn Type', 'Filled Hrs', 'Filled FTE', 'Open Hrs', 'Open FTE', 'BU', 'Date']]
        df = df[df['Rpt Dept'].notna()]
        
        all_data.append(df)  
        print(f"Loaded: {file_name} ({len(df)} rows)")  
    except Exception as e:  
        print(f"ERROR loading {file_name}: {e}")

# Combine all sites into one DataFrame  
pc_all = pd.concat(all_data, ignore_index=True)  
print(f"\nTotal combined rows: {len(combined_df)}")  
print(f"Sites loaded: {combined_df['site'].unique()}")  


Most recent file for LENOX: LENOX POSNEXEC LENOX_EXEC 2026-10-24.xlsx (date: 2026-10-24)
Loaded: LENOX POSNEXEC LENOX_EXEC 2026-10-24.xlsx (5149 rows)

Most recent file for MEETH: MEETH POSNEXEC MEETH_EXEC 2026-10-24.xlsx (date: 2026-10-24)
Loaded: MEETH POSNEXEC MEETH_EXEC 2026-10-24.xlsx (613 rows)

Most recent file for GVCCC: GVCCC POSNEXEC GVCCC_EXEC 2026-10-24.xlsx (date: 2026-10-24)
Loaded: GVCCC POSNEXEC GVCCC_EXEC 2026-10-24.xlsx (488 rows)

Total combined rows: 6259
Sites loaded: <StringArray>
['LENOX', 'MEETH', 'GVCCC']
Length: 3, dtype: str


In [18]:
display(pc_all)

,Rpt Dept,Rpt Dept Desc,Job Code,Jobcode Title,Job Function / Family,Posn Number,Posn Status,Filled/ Open,Incumbent Name,Emplid,Incumbent Status,STD Hrs,Reg/ Temp,Posn Type,Filled Hrs,Filled FTE,Open Hrs,Open FTE,BU,Date
0,15600000.0,Hospital Administration,103442.0,"VP, Marketing",103442.0,10115909.0,A,Filled,"Butler,Mary M",163051.0,A,37.5,R,Full-time,18.75,0.5,0.0,0.0,LENOX,2026-10-24
1,15600000.0,Hospital Administration,107015.0,Site CMIO,107015.0,10064351.0,A,Filled,"Husk,Gregg A",182292.0,A,37.5,R,Full-time,37.50,1.0,0.0,0.0,LENOX,2026-10-24
2,15600000.0,Hospital Administration,200118.0,Associate Biostatistician,200118.0,81032343.0,A,Filled,"Leung,Tungming",225558.0,A,37.5,R,Full-time,18.75,0.5,0.0,0.0,LENOX,2026-10-24
3,15600000.0,Hospital Administration,200222.0,"AVP, IT&S Strategic Planning",200222.0,10050550.0,A,Filled,"Lyons,Colleen M",187683.0,A,37.5,R,Full-time,33.75,0.9,0.0,0.0,LENOX,2026-10-24
4,15600000.0,Hospital Administration,200222.0,"AVP, IT&S Strategic Planning",200222.0,10050550.0,A,Filled,"Lyons,Colleen M",187683.0,A,37.5,R,Full-time,0.00,0.0,0.0,0.0,LENOX,2026-10-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6245,74002598.0,EXE Home,204011.0,"Vice President, Operations",204011.0,81049465.0,A,Open,NaN,NaN,NaN,37.5,R,Full-time,0.00,0.0,0.0,0.0,GVCCC,2026-10-24
6246,74002598.0,EXE Home,204038.0,"Vice President, Hospital Operations, Northwell...",204038.0,81098433.0,A,Open,NaN,NaN,NaN,37.5,R,Full-time,0.00,0.0,0.0,0.0,GVCCC,2026-10-24
6247,74002598.0,EXE Home,204050.0,"VP & Chief Medical Officer, Hospital",204050.0,81087267.0,A,Open,NaN,NaN,NaN,37.5,R,Full-time,0.00,0.0,37.5,1.0,GVCCC,2026-10-24
6248,74002598.0,EXE Home,115816.0,"VP, CMO, NHGV",115816.0,81092275.0,A,Filled,"Muckey,Erin",620874.0,A,37.5,R,Full-time,0.00,0.0,0.0,0.0,GVCCC,2026-10-24


In [21]:
display(mtd_ldr_combined.columns.to_list())


['unnamed_0',
 'unnamed_1',
 'gl_deptid',
 'description',
 'jobcode',
 'descr',
 'name',
 'emplid',
 'hr_deptid',
 'description_1',
 'fiscal_year',
 'accounting_period',
 'pay_end_date',
 'on_off_cycle',
 'check_nbr',
 'check_dt',
 'full_part_time',
 'union_code',
 'reg_dollars',
 'ot_dollars',
 'hol_wkd_dollars',
 'payout_dollars',
 'other_dollars',
 'non_prod_dollars',
 'total_dollars',
 'regular_hours',
 'ot_hours',
 'hol_wkd_hours',
 'non_prod_hours',
 'other_hours',
 'payout_hours',
 'total_hours',
 'report_date',
 'bu',
 'issue_identifier']

In [ ]:
# filter to selected columns
mtd_ldr_f = mtd_ldr_combined[['report_date', 'bu', 'gl_deptid', 'description',  'hr_deptid', 'description_1','jobcode', 'descr', 'emplid', 'regular_hours', 'ot_hours', 
                              'hol_wkd_hours', 'non_prod_hours', 'other_hours', 'payout_hours', 'total_hours']]
mtd_ldr_c = mtd_ldr_f[mtd_ldr_f['gl_deptid'].notna()]


std_hrs = pc_all[['Emplid', 'STD Hrs']]
std_hrs = std_hrs[(std_hrs['STD Hrs'] >= 35) & (std_hrs['STD Hrs'] <= 40)]

mtd_ldr_sh = pd.merge(mtd_ldr_c, std_hrs, how='left', left_on='emplid', right_on='Emplid')





def get_working_days(year, month):  
    """Count weekdays (Mon-Fri) in a given month/year."""  
    start = pd.Timestamp(year=year, month=month, day=1)  
    end = start + pd.offsets.MonthEnd(0)  
    all_days = pd.date_range(start, end, freq='D')  
    working_days = all_days[all_days.weekday < 5]  # Mon=0 ... Fri=4  
    return len(working_days)


# ---------------- config ----------------  
days_per_week = 5  # adjust if any employees work a different schedule

# ---------------- prep ----------------  
df['report_date'] = pd.to_datetime(df['report_date'])

# cache working days per month (faster than recalculating row-by-row)  
df['year_month'] = df['report_date'].dt.to_period('M')

unique_periods = df['year_month'].unique()  
working_days_map = {  
    period: get_working_days(period.year, period.month)  
    for period in unique_periods  
}  
df['working_days'] = df['year_month'].map(working_days_map)

# ---------------- calculate FTE ----------------  
# assumes standard_hours is a WEEKLY value (e.g., 40, 37.5, 20)  
df['standard_daily_hours'] = df['standard_hours'] / days_per_week  
df['standard_monthly_hours'] = df['standard_daily_hours'] * df['working_days']  
df['fte'] = df['hours'] / df['standard_monthly_hours']

# ---------------- cleanup (optional) ----------------  
df = df.drop(columns=['year_month'])

print(df[['report_date', 'hours', 'standard_hours', 'working_days', 'standard_monthly_hours', 'fte']].head())  







# CALC STD HOURS
mtd_ldr_sh['reg_ftes'] = (mtd_ldr_sh['total_hours'] - mtd_ldr_sh['ot_hours'])/mtd_ldr_sh['STD Hrs']
mtd_ldr_sh['ot_ftes'] = mtd_ldr_sh['ot_hours']/mtd_ldr_sh['STD Hrs']
mtd_ldr_sh['total_ftes'] = mtd_ldr_sh['total_hours']/mtd_ldr_sh['STD Hrs']

display(mtd_ldr_sh)
# check totals
mtd_ldr_sh.groupby('bu')[['reg_ftes', 'ot_ftes', 'total_ftes']].sum()

,report_date,bu,gl_deptid,description,hr_deptid,description_1,jobcode,descr,emplid,regular_hours,...,hol_wkd_hours,non_prod_hours,other_hours,payout_hours,total_hours,Emplid,STD Hrs,reg_ftes,ot_ftes,total_ftes
0,2026-01-31,GVCCC,74002500.0,Administration,30003001.0,Nyack Hospital,111637.0,Medical Director,225051.0,6.75,...,0.0,0.00,0.00,0.0,6.75,225051.0,37.5,0.18,0.000000,0.180000
1,2026-01-31,GVCCC,74002500.0,Administration,30003001.0,Nyack Hospital,111637.0,Medical Director,225051.0,0.00,...,0.0,6.75,6.75,0.0,6.75,225051.0,37.5,0.18,0.000000,0.180000
2,2026-01-31,GVCCC,74002500.0,Administration,30003001.0,Nyack Hospital,111637.0,Medical Director,225051.0,6.75,...,0.0,0.00,0.00,0.0,6.75,225051.0,37.5,0.18,0.000000,0.180000
3,2026-01-31,GVCCC,74002500.0,Administration,74002500.0,Administration,202642.0,Product Services & Management Analyst,605165.0,45.00,...,0.0,37.50,37.50,0.0,82.50,605165.0,37.5,2.20,0.000000,2.200000
4,2026-01-31,GVCCC,74002500.0,Administration,74002500.0,Administration,202642.0,Product Services & Management Analyst,605165.0,75.00,...,0.0,-7.50,-7.50,0.0,67.50,605165.0,37.5,1.80,0.000000,1.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134326,2026-10-17,MEETH,29692010.0,Sup Svc-Security,15692010.0,Sup Svc-Security,904094.0,"Senior Director, Safety",139489.0,8.25,...,0.0,0.00,0.00,0.0,8.25,139489.0,37.5,0.22,0.000000,0.220000
134327,2026-10-17,MEETH,29692010.0,Sup Svc-Security,29600000.0,Hospital Administration,201608.0,"Senior Manager, Security",251890.0,82.50,...,0.0,0.00,0.00,0.0,82.50,251890.0,37.5,2.20,0.000000,2.200000
134328,2026-10-17,MEETH,29692010.0,Sup Svc-Security,29600000.0,Hospital Administration,201608.0,"Senior Manager, Security",251890.0,82.50,...,0.0,0.00,0.00,0.0,82.50,251890.0,37.5,2.20,0.000000,2.200000
134329,2026-10-17,MEETH,29693000.0,Sup Svc-Laundry and Linen,29693000.0,Sup Svc-Laundry and Linen,108179.0,Washer,157292.0,75.00,...,0.0,0.00,0.00,0.0,75.00,157292.0,37.5,2.00,0.000000,2.000000


,reg_ftes,ot_ftes,total_ftes
bu,,,
GVCCC,9964.665010,655.967067,10620.632076
LENOX,135729.162650,9280.124419,145009.287069
MEETH,16184.497181,667.328662,16851.825843
